## Installing SDK

To make your journey from fetching data to submitting predictions as smooth as possible, Synnax Lab has created an SDK that's as effortless as a Sunday morning. Let’s install it and get started!

## Imports

In [ ]:
# Importing necessary libraries
import time
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn import linear_model

reg = linear_model.Lasso()

# Importing Synnax Lab SDK Client
from synnax_lab_sdk.client import SynnaxLabClient

## Fetching Datasets

Above script will create a synnax-lab folder in the current working directory where all the downloaded datasets will be stored.
`files` object is a dictionary with files names and their respective paths.

In [ ]:
synnax_lab_client = SynnaxLabClient(api_key = "8c7100ca-f6cc-48e3-8dd6-35b100edd081:Dv0FDMoDKGGdQUDvYILyjB64R5bgSA3gXYAUv9bymkCnI0n91zP3Gx6aHEZ9Hk88rOW7wYA3Ggue-GHXQGq6hQ")


In [ ]:

files = synnax_lab_client.get_datasets(with_macro_data=False)

100%|██████████| 30.5M/30.5M [00:01<00:00, 19.6MB/s]


You have 13 hours, 6 minutes, and 25 seconds remaining until 01/03/2026 23:59:59 to train and submit your predictions for this dataset


In [ ]:
files

{'dataset_date': '2026-03-01',
 'x_train_path': 'synnax-data/datasets/X_train.csv',
 'targets_train_path': 'synnax-data/datasets/targets_train.csv',
 'x_forward_looking_path': 'synnax-data/datasets/X_forward_looking.csv',
 'sample_submission_path': 'synnax-data/datasets/sample_submission.csv',
 'data_dictionary_path': 'synnax-data/datasets/data_dictionary.txt',
 'macro_train_path': None,
 'macro_forward_looking_path': None}

## Dataset Structure 📂

<pre>
📂 synnax-data
│   └── 📂 datasets
│       ├── 📜 data_dictionary.txt
│       ├── 📊 macro_forward_looking.csv
│       ├── 📈 macro_train.csv
│       ├── 📝 sample_submission.csv
│       ├── 🎯 targets_train.csv
│       ├── 🔮 X_forward_looking.csv
│       └── 📚 X_train.csv
</pre>

### 📋 Description:
The datasets subdirectory includes everything you need:

- **`X_train.csv`**: Your training data with financial features.
- **`targets_train.csv`**: The targets you're predicting in training.
- **`X_forward_looking.csv`**: The test data where you’ll make your predictions.
- **`macro_train.csv`** & **`macro_forward_looking.csv`**: Macroeconomic data to enrich your model.
- **`sample_submission.csv`**: Shows you how to format your predictions for submission.



## Loading Dataset

In [ ]:
X_train = pd.read_csv(files['x_train_path'])  # Training features
X_forward_looking = pd.read_csv(files['x_forward_looking_path'])  # Test features
targets_train = pd.read_csv(files['targets_train_path'])  # Training targets
# macro_train = pd.read_csv(files['macro_train_path'])  # Historical macroeconomic data
# macro_forward_looking = pd.read_csv(files['macro_forward_looking_path'])  # Future macroeconomic data

# Clean column names in macroeconomic datasets by removing unsupported characters (mainly spaces)
# macro_train = macro_train.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))
# macro_forward_looking = macro_forward_looking.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))

Let's take a quick look at the first few rows of our training data.

In [ ]:
X_train.head()

,company_id,country_code,sector,industry,Q4_feature_1,Q4_feature_2,Q4_feature_3,Q4_feature_4,Q4_feature_5,Q4_feature_6,...,Q1_feature_23,Q1_feature_24,Q1_feature_25,Q1_feature_26,Q1_feature_27,Q1_feature_28,Q1_feature_29,Q1_feature_30,Q1_feature_31,Q1_feature_32
0,693qc9aLgNddCrQmXnmUod,TW,Technology,Electronics & Computer Distribution,33671.382548,1092.886004,1165.739444,648.418425,239.363586,1934.701374,...,42464.410423,0.493346,28418.643275,3819.802051,15045.636588,25635.178503,15379.767629,63324.050715,29894.000438,1441.512774
1,eLLnpRxs6mX4YpeDDmnR8V,CA,Basic Materials,Other Industrial Metals & Mining,0.494279,-0.473187,-0.470896,-0.854963,1.445617,0.492414,...,2.744096,0.493346,0.493346,0.493346,2.763354,0.655881,9.176036,11.473688,2.744219,0.881107
2,2fEAN3aYGDxBpCdv5CA69j,TW,Basic Materials,Building Materials,1037.834035,385.470856,422.891504,370.229942,50.915431,283.019226,...,3690.347162,0.493346,2073.385001,1442.115653,1126.857257,976.597458,15173.251341,20938.831138,3881.636105,15839.265822
3,Febe4QhRvncnJy4woudLYR,US,Basic Materials,Other Industrial Metals & Mining,3565.193477,-1312.815282,-411.267150,-1619.727767,0.493346,-637.930333,...,12356.979031,0.493346,5926.020656,39459.790731,4472.123111,3029.874207,46789.866112,108512.681829,42798.693406,63885.466138
4,DsMf47eqAGyDc56ajQBvby,CA,Basic Materials,Silver,0.712102,-37.890121,-37.688626,-33.694073,30.686754,0.273415,...,18.687758,0.493346,0.493346,2.604737,17.393281,3.560264,1786.230480,1810.194045,3.898928,1096.950490


## Checking Categorical Columns

In [ ]:
cat_cols = X_train.select_dtypes(include='object').columns
cat_cols

Index(['company_id', 'country_code', 'sector', 'industry'], dtype='object')

Don't miss the two columns containing dates! 📅

In [ ]:
# Check the number of unique values in the categorical columns
X_train[cat_cols[1:]].nunique()

,0
country_code,78
sector,11
industry,138


# Process data

In [ ]:
# Combine training and test data for consistent encoding
data = pd.concat([X_train, X_forward_looking], axis=0)

## Extract features from dates

## Encode categorical variables

In [ ]:
# Encode categorical columns using LabelEncoder
for col in cat_cols:
    data[col] = LabelEncoder().fit_transform(data[col])

There are multiple ways to encode categoric variables and they should be selected given the model you will be using and the nature of the data.

Tree-based models can take advantage of any kind of encoding, while for linear models `LabelEncoder` might not be a viable option because it makes the categories ordinal.

Try implementing:
- one-hot-encoding
- label-encoding
- frequency-encoding
- mean-target-encoding

to find the best option for your model.

## Handling Missing Values

### Define Function

In [ ]:
def fill_missing_with_mean(df):
    """
    Fill missing values with the mean of each column.
    """
    for col in df.columns:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mean())
    return df

### Apply Function to Datasets

In [ ]:
data = fill_missing_with_mean(data)


In [ ]:
# drop columns with all missing (sometimes that happens)
for col in data:
    if data[col].isnull().all():
        print(col)
        data.drop(col, axis=1, inplace=True)

Missing values imputation is a tricky process. While replacing NaNs with mean of the column technically makes your dataset for modeling, it might populate your *ground truth* training data with a lot of faulty values. In many cases mean of the column will not be a right choice.

Consider:
- median
- mode
- KNN imputer
- verstack.NaNImputer
- etc.

In [ ]:
# Split data back into training and test sets
X_train = data[:X_train.shape[0]]
X_forward_looking = data[X_train.shape[0]:]

## Dropping `companyId`

In [ ]:
X_train = X_train.drop('company_id', axis=1)
X_forward_looking = X_forward_looking.drop('company_id', axis=1)
targets_train = targets_train.drop('company_id', axis=1)
# macro_train = macro_train.drop('companyId', axis=1)
# macro_forward_looking = macro_forward_looking.drop('companyId', axis=1)

## Training

### Example Hyperparameters

In [ ]:
# define starter parameters for LGBMRegressor
params = {
    'learning_rate': 0.01,
    'num_leaves': 250,
    'feature_fraction': 0.5,
    'bagging_fraction': 0.9,
    'verbosity': -1,
    'random_state': 42,
    'device_type': 'cpu',
    'objective': 'regression',
    'metric': 'l2',
    'num_threads': 10,
    'lambda_l1': 0.5,
    'n_estimators': 100
    }

### Train the Model

We will use the same type of model with fixed parameters to predict each of the 17 targets. `sklearn.multioutput.MultiOutputRegressor` will simplify our code and rather than producing code to train 17 independent models, we can use `MultiOutputRegressor` to do it for us in a convenient sklearn-style one-liner.

But remember, each target, even though it represents financial indicators from one company and one time-period, may have different dependencies and even may require different types of models. So try different strategies to improve your score.

In [ ]:
# from sklearn.decomposition import PCA
# import matplotlib.pyplot as plt
# import numpy as np

# pca = PCA().fit(X_train)

# plt.figure(figsize=(10,4))
# plt.plot(np.cumsum(pca.explained_variance_ratio_), color='k', lw=2)
# plt.xlabel('Number of components')
# plt.ylabel('Total explained variance')
# plt.xlim(0, 10)
# plt.yticks(np.arange(0, 1.1, 0.1))
# plt.show();

In [ ]:
# def cor_top(train, targets, target, num):
#   return list(pd.Series({col: abs(targets[target].corr(train[col])) for col in train.columns}).sort_values(ascending=False)[:num].index)

def cor_top(train, targets, target, num):
  correlations = {}
  for col in train.columns:
    correlations[col] = abs(targets[target].corr(train[col]))
  correlations_series = pd.Series(correlations)
  sorted_correlations = correlations_series.sort_values(ascending=False)
  return list(sorted_correlations[:num].index)

In [ ]:
from sklearn.linear_model import HuberRegressor, TweedieRegressor, QuantileRegressor, PassiveAggressiveRegressor, TheilSenRegressor, LassoLarsCV, OrthogonalMatchingPursuitCV
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor

# from autosklearn.regression import AutoSklearnRegressor


# X_train = TSNE(n_components=2).fit_transform(X_train)
# X_forward_looking = TSNE(n_components=2).fit_transform(X_forward_looking)
# X_train = PCA(n_components=100).fit_transform(X_train)
# X_forward_looking = PCA(n_components=100).fit_transform(X_forward_looking)

def train_and_predict_cor(model, num):
  preds = {}
  for target in targets_train:
    top = cor_top(X_train, targets_train, target, num)
    model.fit(X_train[top], targets_train[target])
    preds[target] = model.predict(X_forward_looking[top])
  sample_submission = pd.read_csv(files['sample_submission_path'])
  for target in preds:
    sample_submission[target] = preds[target]
  return sample_submission

def submit_cor(submission_path, sample_submission):
  sample_submission.to_csv(submission_path, index=False)
  synnax_lab_client.submit_predictions(files["dataset_date"], submission_path)

def run_multiple_cor(models):
  for cor_top_cnt in range(2, 22):
    for model in models:
      print(model.__name__)
      try:
        sample_submission = train_and_predict_cor(model(random_state=408), cor_top_cnt)
      except:
        sample_submission = train_and_predict_cor(model(), cor_top_cnt)
      submission_path = f'{model.__name__}.csv'
      submit_cor(submission_path, sample_submission)

# TODO: LAMA, HungaBunga, Auto-sklearn для перебора моделей; подобрать параметры --- grid-search, Auto ML, Optuna
def train_and_predict(model):
  preds = {}
  for target in targets_train:
    model.fit(X_train, targets_train[target])
    preds[target] = model.predict(X_forward_looking)
  sample_submission = pd.read_csv(files['sample_submission_path'])
  for target in preds:
    sample_submission[target] = preds[target]
  return sample_submission

def submit(submission_path, sample_submission):
  sample_submission.to_csv(submission_path, index=False)
  synnax_lab_client.submit_predictions(files["dataset_date"], submission_path)

def run_multiple(models):
  for model in models:
    print(model.__name__)
    sample_submission = train_and_predict(model())
    submission_path = f'{model.__name__}.csv'
    submit(submission_path, sample_submission)

def run_multiple_long(models):
  for model in models:
    print(model.__name__)
    sample_submission = train_and_predict(model(random_state=408))
    submission_path = f'{model.__name__}.csv'
    submit(submission_path, sample_submission)

run_multiple([LassoLarsCV, OrthogonalMatchingPursuitCV])
run_multiple_cor([TheilSenRegressor])
run_multiple_cor([RandomForestRegressor])
#run_multiple_long([RandomForestRegressor, TheilSenRegressor])


LassoLarsCV
Uploading submission e11e221c-bc51-4219-92cf-dd59c5e69ad5 for 2026-03-01...


100%|██████████| 4.15M/4.15M [00:01<00:00, 2.55MB/s]


Uploaded submission e11e221c-bc51-4219-92cf-dd59c5e69ad5
OrthogonalMatchingPursuitCV
Uploading submission 7e0642b4-1741-41f0-92b3-84b001c5517c for 2026-03-01...


100%|██████████| 4.17M/4.17M [00:01<00:00, 3.19MB/s]


Uploaded submission 7e0642b4-1741-41f0-92b3-84b001c5517c
TheilSenRegressor
Uploading submission 3c5844b2-9e6b-44fa-81b3-9d79dd9f824b for 2026-03-01...


...

Uploaded submission 4b99763c-8234-4c03-882d-d44edf6b1c62
RandomForestRegressor
Uploading submission cc3389ee-4597-4a92-86cc-6674100b404e for 2026-03-01...


100%|██████████| 4.13M/4.13M [00:01<00:00, 3.57MB/s]


Uploaded submission cc3389ee-4597-4a92-86cc-6674100b404e
RandomForestRegressor
Uploading submission b966b683-b413-43a8-a76a-a0605d7a0544 for 2026-03-01...


100%|██████████| 4.13M/4.13M [00:01<00:00, 3.29MB/s]


Uploaded submission b966b683-b413-43a8-a76a-a0605d7a0544
RandomForestRegressor


### Make Predictions

## Submitting Predictions

### Save Submission File

In [ ]:
# Save the updated submission file
# submission_path = 'submission_lassolarscv.csv'
# sample_submission.to_csv(submission_path, index=False)

### Submit the Predictions

In [ ]:
# synnax_lab_client.submit_predictions(files["dataset_date"], submission_path)
# time.sleep(5) # 5 seconds to process submission before running next cell

## Check confidence score (validation)

Submissions require some time to get processed in the backend, so in case running the below function does not return the `confidenceScore` right away, give it a few seconds and rerun. A `confidenceScore` is there when the submission `status: 'Processed'`

In [ ]:
submissions = synnax_lab_client.get_past_submissions()

## Congratulations! 🥳 You’ve Made Your First Submission!

This was a basic pipeline. Now, it's time to level up! 🚀 Use the macroeconomic data to make your model more robust. Experiment with different models, tweak them, and maybe even try some neural networks. 🧠💥

Data science is like cooking. There are endless recipes to try. So, spice things up, preprocess like a pro, and get those scores soaring! 🌟👨‍🍳👩‍🍳

Good luck, and may the data be ever in your favor! 🍀📈

# P.S.

Above pipeline is a simple example of how to get started with synnax-lab-sdk and arrive at your first submission.

To improve your scores look into:
1. Macroeconomic data
2. Experiment with other categoric variables encoding options (try individual mean-target-encoding for each target)
3. Deal with outliers
4. More advanced missing values imputation options
5. Different models, individual models for each tartet, hyperparameters tuning
6. Models ensembling (if using different models, make sure you have appropriate processing for each model)
7. Feature selection: not all features in X_train, macro_train can be useful. Try to remove some of them.
8. And of course your creativity. We're confident that your individual approach can beat anything we have layed out in our short tutorial

In [ ]:
from datetime import date
today_str = date.today().isoformat()  # "YYYY-MM-DD"
today_submissions = [s for s in submissions if s.get("uploadedAt", "").startswith(today_str)]
print(today_submissions)